
## Unified ASR Audio Pipeline: TTS, Ambulance Noise and Whisper

This notebook applies the same ASR pipeline to either the full dataset
or the held-out test set.

Pipeline:

`text_cleaned → TTS → very high ambulance noise → Whisper ASR → WER`

## 1. Environment Setup

This section installs and imports the packages required for text-to-speech,
audio processing, Whisper transcription, and WER calculation.

In [ ]:
# Install required packages for TTS, audio processing, ASR, and WER calculation
!pip install -q edge-tts nest_asyncio pydub openai-whisper jiwer tqdm

# Install ffmpeg for audio conversion
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [ ]:
import os
import torch
import whisper
import pandas as pd
import edge_tts
import asyncio
import nest_asyncio

from tqdm import tqdm
from jiwer import wer
from pydub import AudioSegment
from pydub.generators import Sine, WhiteNoise
from IPython.display import Audio

# Allow async code to run properly in Colab/Jupyter
nest_asyncio.apply()

print("Imports completed successfully")
print("CUDA available:", torch.cuda.is_available())

Imports completed successfully
CUDA available: False


## 2. Experiment Configuration

The notebook can process either the full dataset or the held-out test set.
The selected mode automatically determines the input file, output paths,
audio folders, and expected number of rows.

In [ ]:
# Choose which dataset to process
dataset_mode = "full"   # options: "full" or "test"

## 3. File Paths and Audio Folders

Input, progress, final output, and audio folder names are generated
automatically according to the selected dataset mode.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd

drive_output_dir = "/content/drive/MyDrive/MedAlert_ASR_outputs"
os.makedirs(drive_output_dir, exist_ok=True)

if dataset_mode == "full":
    input_path = "asr_full_dataset_8556_from_text_cleaned.csv"
    dataset_size = 8556

elif dataset_mode == "test":
    input_path = "asr_full_test_set_1284_complete_cases.csv"
    dataset_size = 1284

else:
    raise ValueError("dataset_mode must be 'full' or 'test'")

progress_path = os.path.join(
    drive_output_dir,
    "asr_results_progress_" + str(dataset_size) + "_very_high_text_cleaned.csv"
)

final_path = os.path.join(
    drive_output_dir,
    "ems_asr_noisy_dataset_" + str(dataset_size) + "_very_high_text_cleaned.csv"
)

clean_audio_dir = (
    "clean_audio_" + str(dataset_size) + "_text_cleaned_very_high"
)

noisy_audio_dir = (
    "noisy_audio_" + str(dataset_size) + "_text_cleaned_very_high"
)

os.makedirs(clean_audio_dir, exist_ok=True)
os.makedirs(noisy_audio_dir, exist_ok=True)

print("Dataset mode:", dataset_mode)
print("Expected rows:", dataset_size)
print("Input file:", input_path)
print("Input file exists:", os.path.exists(input_path))
print("Progress path:", progress_path)
print("Final path:", final_path)
print("Clean audio folder:", clean_audio_dir)
print("Noisy audio folder:", noisy_audio_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset mode: full
Expected rows: 8556
Input file: asr_full_dataset_8556_from_text_cleaned.csv
Input file exists: True
Progress path: /content/drive/MyDrive/MedAlert_ASR_outputs/asr_results_progress_8556_very_high_text_cleaned.csv
Final path: /content/drive/MyDrive/MedAlert_ASR_outputs/ems_asr_noisy_dataset_8556_very_high_text_cleaned.csv
Clean audio folder: clean_audio_8556_text_cleaned_very_high
Noisy audio folder: noisy_audio_8556_text_cleaned_very_high


## 4. Load and Validate the Selected Dataset

The selected CSV file is loaded and checked for the expected number of rows,
missing texts, empty texts, and the distribution of text variants.

In [ ]:
# ============================================================
# Load and validate the selected ASR dataset
# ============================================================

# Check that the selected input file exists before loading
print("Input file:", input_path)
print("Input file exists:", os.path.exists(input_path))

if not os.path.exists(input_path):
    raise FileNotFoundError(
        "The selected input file was not found: " + input_path
    )

# Load the selected ASR dataset
df_asr = pd.read_csv(input_path)

# Reset index for easier audio file naming
df_asr = df_asr.reset_index(drop=True)

# Define the column that will be converted to speech
text_col = "text_cleaned"

# Check that the selected text column exists
print("\nDataset mode:", dataset_mode)
print("Using text column:", text_col)
print("Column exists:", text_col in df_asr.columns)

if text_col not in df_asr.columns:
    raise KeyError(
        "The selected text column was not found: " + text_col
    )

# Print basic dataset information
print("\nSelected ASR dataset shape:", df_asr.shape)
print("Expected number of rows:", dataset_size)
print("Actual number of rows:", len(df_asr))
print("Number of source cases:", df_asr["source_case_id"].nunique())

# Verify that the loaded file matches the selected dataset mode
print(
    "Row count matches selected mode:",
    len(df_asr) == dataset_size
)

# Check missing or empty cleaned texts
print(
    "\nMissing values in text_cleaned:",
    df_asr[text_col].isna().sum()
)

print(
    "Empty texts in text_cleaned:",
    (
        df_asr[text_col]
        .fillna("")
        .astype(str)
        .str.strip()
        == ""
    ).sum()
)

# Print the distribution of text variants
print("\nText variants:")
print(df_asr["text_variant"].value_counts())

# Display the first rows with original and cleaned text
df_asr[
    [
        "source_case_id",
        "text_variant",
        "text",
        "text_cleaned"
    ]
].head()

Input file: asr_full_dataset_8556_from_text_cleaned.csv
Input file exists: True

Dataset mode: full
Using text column: text_cleaned
Column exists: True

Selected ASR dataset shape: (8556, 15)
Expected number of rows: 8556
Actual number of rows: 8556
Number of source cases: 2139
Row count matches selected mode: True

Missing values in text_cleaned: 0
Empty texts in text_cleaned: 0

Text variants:
text_variant
professional_complete                 2139
brief_radio_missing_details           2139
patient_reported_uncertain            2139
distracted_or_disorganized_handoff    2139
Name: count, dtype: int64


,source_case_id,text_variant,text,text_cleaned
0,32822973,professional_complete,We are transporting an 80-year-old female with...,We are transporting an 80-year-old female with...
1,32822973,brief_radio_missing_details,80-year-old female with recent sepsis and intu...,80-year-old female with recent sepsis and intu...
2,32822973,patient_reported_uncertain,We’re bringing in an 80-year-old female with a...,We’re bringing in an 80-year-old female with a...
3,32822973,distracted_or_disorganized_handoff,80-year-old female with history of Sjogren’s a...,80-year-old female with history of Sjogren’s a...
4,31591237,professional_complete,We are transporting a 69-year-old male with a ...,We are transporting a 69-year-old male with a ...


In [ ]:
# Check that each source case has exactly 4 text variants
variant_count_per_case = df_asr.groupby("source_case_id")["text_variant"].nunique()

print(variant_count_per_case.describe())

print("\nNumber of cases that do not have exactly 4 variants:")
print((variant_count_per_case != 4).sum())

count    2139.0
mean        4.0
std         0.0
min         4.0
25%         4.0
50%         4.0
75%         4.0
max         4.0
Name: text_variant, dtype: float64

Number of cases that do not have exactly 4 variants:
0


In [ ]:
# ============================================================
# Display the selected sample before TTS
# ============================================================

sample_index = 0

print("Dataset mode:", dataset_mode)
print("Sample index:", sample_index)
print("Text variant:", df_asr.loc[sample_index, "text_variant"])

print("\nCleaned report text used for TTS:")
print(df_asr.loc[sample_index, text_col])

print(
    "\nNumber of words:",
    len(str(df_asr.loc[sample_index, text_col]).split())
)

Dataset mode: full
Sample index: 0
Text variant: professional_complete

Cleaned report text used for TTS:
We are transporting an 80-year-old female with a history of Sjogren’s syndrome, moderate mitral regurgitation, and recent sepsis from C. diff colitis complicated by respiratory failure requiring intubation. She presents with worsening shortness of breath and cough for one day. Vitals are temperature 100.2, heart rate 95, respiratory rate 28, blood pressure 99/47, and oxygen saturation 100% on 2 liters nasal cannula. She denies chest pain, nausea, vomiting, or abdominal pain. Patient is stable but tachypneic.

Number of words: 76


## 5. Text-to-Speech Generation

Each cleaned EMS report is converted into speech using a male Edge-TTS voice.
The speech is accelerated to simulate a fast ambulance radio handoff.

In [ ]:
async def text_to_speech_male_async(text, output_path, speed=1.3):
    """
    Convert a text report into speech using a male neural voice.
    Then speed up the audio to simulate ambulance radio speech.
    """
    text = str(text)

    if len(text.strip()) == 0:
        text = "Empty report"

    temp_path = output_path.replace(".mp3", "_temp.mp3")
    voice = "en-US-GuyNeural"

    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(temp_path)

    audio = AudioSegment.from_file(temp_path)
    faster_audio = audio.speedup(playback_speed=speed)
    faster_audio.export(output_path, format="mp3")

    if os.path.exists(temp_path):
        os.remove(temp_path)


def text_to_speech(text, output_path, speed=1.3):
    """
    Wrapper function for generating TTS audio in Colab.
    """
    loop = asyncio.get_event_loop()
    loop.run_until_complete(
        text_to_speech_male_async(text, output_path, speed)
    )

In [ ]:
# ============================================================
# Generate one clean TTS audio sample
# ============================================================

sample_index = 0

# Select one cleaned report from the selected dataset
sample_text = df_asr.loc[sample_index, text_col]

# Define the output path inside the selected dataset folder
sample_audio_path = os.path.join(
    clean_audio_dir,
    "sample_0_clean_male_fast.mp3"
)

# Convert the cleaned report to speech
text_to_speech(
    sample_text,
    sample_audio_path,
    speed=1.3
)

# Load the audio and calculate its duration
audio = AudioSegment.from_file(sample_audio_path)
duration_seconds = len(audio) / 1000

print("Dataset mode:", dataset_mode)
print("Saved clean audio:", sample_audio_path)
print("Audio duration:", duration_seconds, "seconds")

Dataset mode: full
Saved clean audio: clean_audio_8556_text_cleaned_very_high/sample_0_clean_male_fast.mp3
Audio duration: 27.28 seconds


In [ ]:
# Check whether the generated audio file from text_cleaned exists
print("File path:", sample_audio_path)
print("File exists:", os.path.exists(sample_audio_path))
print("File size:", os.path.getsize(sample_audio_path), "bytes")

File path: clean_audio_8556_text_cleaned_very_high/sample_0_clean_male_fast.mp3
File exists: True
File size: 109581 bytes


In [ ]:
# Play the generated clean audio file
Audio(sample_audio_path)

In [ ]:
from google.colab import files

# Download the clean sample audio file to the computer
files.download(sample_audio_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Very High Ambulance Noise Simulation

A synthetic ambulance siren and white background noise are added to the
clean speech. The siren gain is set to -6 dB and the background noise
gain is set to -18 dB.

In [ ]:
def create_siren_noise(duration_ms):
    """
    Create a synthetic ambulance-like siren noise.
    The siren alternates between two frequencies.
    """
    siren = AudioSegment.silent(duration=0)

    high_tone = Sine(900).to_audio_segment(duration=400)
    low_tone = Sine(600).to_audio_segment(duration=400)

    while len(siren) < duration_ms:
        siren += high_tone
        siren += low_tone

    # Trim the siren to match the exact duration of the speech audio
    siren = siren[:duration_ms]

    return siren

In [ ]:
def add_ambulance_noise(clean_audio_path, noisy_audio_path):
    """
    Add very high ambulance siren and background noise to a clean audio file.
    The noisy audio is saved as a WAV file.
    """
    # Load the clean speech audio
    speech = AudioSegment.from_file(clean_audio_path)

    # Get the duration of the speech audio in milliseconds
    duration_ms = len(speech)

    # Create ambulance-like siren and general background noise
    siren = create_siren_noise(duration_ms)
    background_noise = WhiteNoise().to_audio_segment(duration=duration_ms)

    # Apply only very high noise
    siren = siren.apply_gain(-6)
    background_noise = background_noise.apply_gain(-18)

    # Overlay the noise on top of the original speech
    noisy_audio = speech.overlay(background_noise)
    noisy_audio = noisy_audio.overlay(siren)

    # Export as WAV because Whisper works well with WAV files
    noisy_audio.export(noisy_audio_path, format="wav")

In [ ]:
# Define the output path for the noisy audio sample
sample_noisy_audio_path = os.path.join(noisy_audio_dir, "sample_0_noisy.wav")

# Add very high ambulance noise to the clean audio sample
add_ambulance_noise(
    clean_audio_path=sample_audio_path,
    noisy_audio_path=sample_noisy_audio_path
)

# Check whether the noisy audio file was created successfully
print("Saved noisy sample audio:", sample_noisy_audio_path)
print("File exists:", os.path.exists(sample_noisy_audio_path))
print("File size:", os.path.getsize(sample_noisy_audio_path), "bytes")

Saved noisy sample audio: noisy_audio_8556_text_cleaned_very_high/sample_0_noisy.wav
File exists: True
File size: 2406140 bytes


In [ ]:
# Play the noisy audio file
Audio(sample_noisy_audio_path)

In [ ]:
from google.colab import files

# Download the noisy sample audio file to the computer
files.download(sample_noisy_audio_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Whisper ASR Transcription

The noisy ambulance audio is transcribed using the Whisper base model.
WER is then calculated between the original cleaned report and the
resulting ASR transcript.

In [ ]:
# Load Whisper ASR model
# "base" is faster, "small" is usually more accurate but slower
asr_model = whisper.load_model("base")

print("Whisper model loaded successfully")
print("CUDA available:", torch.cuda.is_available())

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 82.1MiB/s]


Whisper model loaded successfully
CUDA available: False


In [ ]:
def transcribe_audio(audio_path):
    """
    Transcribe an audio file using Whisper ASR.
    Returns the ASR transcript as text.
    """
    result = asr_model.transcribe(
        audio_path,
        language="en",
        fp16=torch.cuda.is_available()
    )

    return result["text"].strip()

In [ ]:
# Transcribe the noisy audio sample
sample_asr_text = transcribe_audio(sample_noisy_audio_path)

print("Original cleaned text:")
print(sample_text)

print("\nASR noisy transcript:")
print(sample_asr_text)

print("\nWER:")
print(wer(sample_text, sample_asr_text))

Original cleaned text:
We are transporting an 80-year-old female with a history of Sjogren’s syndrome, moderate mitral regurgitation, and recent sepsis from C. diff colitis complicated by respiratory failure requiring intubation. She presents with worsening shortness of breath and cough for one day. Vitals are temperature 100.2, heart rate 95, respiratory rate 28, blood pressure 99/47, and oxygen saturation 100% on 2 liters nasal cannula. She denies chest pain, nausea, vomiting, or abdominal pain. Patient is stable but tachypneic.

ASR noisy transcript:
We are transporting an 80 year old female with a history of children's syndrome, moderate Michael Rodriguez Haitian, and recent status from Steve Diffle light is complicated by respiratory failure with firing innovation. She presents the first thing short as a weapon call for Monday. White older cabbage from 100.2, heartbeat 95, respiratory rate 28, blood pressure 99.7, an oxygen saturation 100% on two liters of annual. She did a nice c

## 8. Pre-Run Validation

Before processing the full selected dataset, the notebook verifies the
dataset mode, input file, output paths, audio folders, and noise settings.

In [ ]:
# ============================================================
# Verify ASR settings before running the full pipeline
# ============================================================

print("Dataset mode:", dataset_mode)
print("Expected dataset size:", dataset_size)

print("\nInput path:")
print(input_path)
print("Input file exists:", os.path.exists(input_path))

print("\nProgress path:")
print(progress_path)

print("\nFinal path:")
print(final_path)

print("\nClean audio folder:")
print(clean_audio_dir)

print("\nNoisy audio folder:")
print(noisy_audio_dir)

print("\nExpected checks:")
print("Progress path contains dataset size:", str(dataset_size) in progress_path)
print("Final path contains dataset size:", str(dataset_size) in final_path)
print("Clean folder contains dataset size:", str(dataset_size) in clean_audio_dir)
print("Noisy folder contains dataset size:", str(dataset_size) in noisy_audio_dir)

print("Progress path contains very_high:", "very_high" in progress_path)
print("Final path contains very_high:", "very_high" in final_path)

Dataset mode: full
Expected dataset size: 8556

Input path:
asr_full_dataset_8556_from_text_cleaned.csv
Input file exists: True

Progress path:
/content/drive/MyDrive/MedAlert_ASR_outputs/asr_results_progress_8556_very_high_text_cleaned.csv

Final path:
/content/drive/MyDrive/MedAlert_ASR_outputs/ems_asr_noisy_dataset_8556_very_high_text_cleaned.csv

Clean audio folder:
clean_audio_8556_text_cleaned_very_high

Noisy audio folder:
noisy_audio_8556_text_cleaned_very_high

Expected checks:
Progress path contains dataset size: True
Final path contains dataset size: True
Clean folder contains dataset size: True
Noisy folder contains dataset size: True
Progress path contains very_high: True
Final path contains very_high: True


## 9. Full ASR Pipeline with Progress Saving

The complete pipeline is applied to every report in the selected dataset.
Progress is saved to Google Drive every 25 processed reports, allowing the
run to resume after a runtime interruption.

In [ ]:
# ============================================================
# Run ASR pipeline for the selected dataset
# with very high noise and progress saving
# ============================================================

# Label saved in the final CSV
# The actual very high noise intensity is defined
# inside add_ambulance_noise()
noise_level = "very_high"

print("Dataset mode:", dataset_mode)
print("Dataset size:", dataset_size)
print("Number of rows to process:", len(df_asr))
print("Noise level:", noise_level)
print("Progress file:", progress_path)

# Load existing progress if the runtime stopped previously
if os.path.exists(progress_path):
    df_progress = pd.read_csv(progress_path)

    # Convert previous results into a list of dictionaries
    asr_results = df_progress.to_dict("records")

    # Store completed indices so they will not run again
    completed_indices = set(
        df_progress["sample_index"].astype(int).tolist()
    )

    print("\nExisting progress file found.")
    print("Completed samples:", len(completed_indices))

    if len(completed_indices) > 0:
        print("Highest completed index:", max(completed_indices))

else:
    # Start a new run
    asr_results = []
    completed_indices = set()

    print("\nNo existing progress file found.")
    print("Starting ASR pipeline from the beginning.")


# Run the pipeline on the selected dataset
for i, row in tqdm(
    df_asr.iterrows(),
    total=len(df_asr),
    initial=len(completed_indices)
):

    # Skip samples already saved in the progress file
    if i in completed_indices:
        continue

    # Get the cleaned report text
    original_text = str(row[text_col])

    # Define audio paths according to the selected dataset
    clean_audio_path = os.path.join(
        clean_audio_dir,
        str(i) + "_clean.mp3"
    )

    noisy_audio_path = os.path.join(
        noisy_audio_dir,
        str(i) + "_noisy.wav"
    )

    try:
        # Step 1: Convert the cleaned text to speech
        text_to_speech(
            text=original_text,
            output_path=clean_audio_path,
            speed=1.3
        )

        # Step 2: Add fixed very high ambulance noise
        add_ambulance_noise(
            clean_audio_path=clean_audio_path,
            noisy_audio_path=noisy_audio_path
        )

        # Step 3: Transcribe the noisy audio using Whisper
        asr_text = transcribe_audio(noisy_audio_path)

        # Step 4: Calculate WER
        sample_wer = wer(original_text, asr_text)

    except Exception as e:
        # Save the failed sample so it can be retried later
        asr_text = ""
        sample_wer = None

        print("\nError in sample:", i)
        print(e)

    # Store the result
    asr_results.append({
        "sample_index": i,
        "original_text_for_asr": original_text,
        "asr_text_noisy": asr_text,
        "noise_level": noise_level,
        "wer": sample_wer,
        "clean_audio_path": clean_audio_path,
        "noisy_audio_path": noisy_audio_path
    })

    # Save progress every 25 newly processed samples
    if len(asr_results) % 25 == 0:
        pd.DataFrame(asr_results).to_csv(
            progress_path,
            index=False
        )

        print("\nProgress saved.")
        print("Total saved results:", len(asr_results))


# Save all progress at the end
df_asr_results = pd.DataFrame(asr_results)

df_asr_results = (
    df_asr_results
    .sort_values("sample_index")
    .reset_index(drop=True)
)

df_asr_results.to_csv(progress_path, index=False)

print("\nASR pipeline completed.")
print("Dataset mode:", dataset_mode)
print("Expected rows:", dataset_size)
print("Total saved results:", len(df_asr_results))
print("Progress file saved to:")
print(progress_path)

Dataset mode: full
Dataset size: 8556
Number of rows to process: 8556
Noise level: very_high
Progress file: /content/drive/MyDrive/MedAlert_ASR_outputs/asr_results_progress_8556_very_high_text_cleaned.csv

Existing progress file found.
Completed samples: 8556
Highest completed index: 8555


17112it [00:00, 27484.46it/s]



ASR pipeline completed.
Dataset mode: full
Expected rows: 8556
Total saved results: 8556
Progress file saved to:
/content/drive/MyDrive/MedAlert_ASR_outputs/asr_results_progress_8556_very_high_text_cleaned.csv


## 10. Load Saved ASR Results

The saved progress file is loaded and sorted according to the original
sample indices before checking for failed reports.

In [ ]:
# ============================================================
# Load saved ASR results
# ============================================================

df_asr_results = pd.read_csv(progress_path)

df_asr_results = (
    df_asr_results
    .sort_values("sample_index")
    .reset_index(drop=True)
)

print("Original dataset rows:", len(df_asr))
print("ASR result rows:", len(df_asr_results))
print(
    "Unique sample indices:",
    df_asr_results["sample_index"].nunique()
)

if len(df_asr) != len(df_asr_results):
    raise ValueError(
        "The original dataset and ASR results have different row counts."
    )

expected_indices = list(range(len(df_asr)))
actual_indices = df_asr_results["sample_index"].astype(int).tolist()

if actual_indices != expected_indices:
    raise ValueError(
        "The ASR sample indices do not match the original dataset order."
    )

print("ASR results loaded and validated successfully.")

Original dataset rows: 8556
ASR result rows: 8556
Unique sample indices: 8556
Saved final file to Google Drive:
/content/drive/MyDrive/MedAlert_ASR_outputs/ems_asr_noisy_dataset_8556_very_high_text_cleaned.csv
Final shape: (8556, 21)


## 11. Failed Sample Detection and Retry

Reports with a missing transcript, an empty transcript, or a missing WER
value are identified and processed again.

In [ ]:
# ============================================================
# Detect failed samples
# ============================================================

failed_samples = df_asr_results[
    (df_asr_results["asr_text_noisy"].isna()) |
    (df_asr_results["asr_text_noisy"].astype(str).str.strip() == "") |
    (df_asr_results["wer"].isna())
]

print("Number of failed samples:", len(failed_samples))

failed_samples

Number of failed samples: 2


,sample_index,original_text_for_asr,asr_text_noisy,noise_level,wer,clean_audio_path,noisy_audio_path
5585,5585,20-year-old male with 5-day swelling and pain ...,NaN,very_high,NaN,clean_audio_8556_text_cleaned_very_high/5585_c...,noisy_audio_8556_text_cleaned_very_high/5585_n...
6778,6778,We’re bringing in a 91-year-old female who rep...,NaN,very_high,NaN,clean_audio_8556_text_cleaned_very_high/6778_c...,noisy_audio_8556_text_cleaned_very_high/6778_n...


In [ ]:
# ============================================================
# Retry only failed samples
# ============================================================

noise_level = "very_high"

if len(failed_samples) == 0:
    print("No failed samples. Retry step is not needed.")

else:
    for idx in failed_samples["sample_index"].astype(int):
        print("Retrying sample:", idx)

        # Verify that the failed index belongs to the selected dataset
        if idx not in df_asr.index:
            print("Sample index is outside the selected dataset:", idx)
            continue

        # Get the cleaned text again
        original_text = str(df_asr.loc[idx, text_col])

        # Define output paths
        clean_audio_path = os.path.join(
            clean_audio_dir,
            str(idx) + "_clean.mp3"
        )

        noisy_audio_path = os.path.join(
            noisy_audio_dir,
            str(idx) + "_noisy.wav"
        )

        try:
            # Step 1: Convert cleaned text to speech again
            text_to_speech(
                text=original_text,
                output_path=clean_audio_path,
                speed=1.3
            )

            # Step 2: Add fixed very high ambulance noise
            add_ambulance_noise(
                clean_audio_path=clean_audio_path,
                noisy_audio_path=noisy_audio_path
            )

            # Step 3: Transcribe using Whisper
            asr_text = transcribe_audio(noisy_audio_path)

            # Step 4: Calculate WER
            sample_wer = wer(original_text, asr_text)

            # Locate the corresponding result row
            row_mask = df_asr_results["sample_index"] == idx

            # Update all relevant values
            df_asr_results.loc[
                row_mask, "original_text_for_asr"
            ] = original_text

            df_asr_results.loc[
                row_mask, "asr_text_noisy"
            ] = asr_text

            df_asr_results.loc[
                row_mask, "noise_level"
            ] = noise_level

            df_asr_results.loc[
                row_mask, "wer"
            ] = sample_wer

            df_asr_results.loc[
                row_mask, "clean_audio_path"
            ] = clean_audio_path

            df_asr_results.loc[
                row_mask, "noisy_audio_path"
            ] = noisy_audio_path

            print("Retry succeeded for sample:", idx)
            print("WER:", sample_wer)

        except Exception as e:
            print("Retry failed again for sample:", idx)
            print(e)

Retrying sample: 5585
Retry succeeded for sample: 5585
WER: 0.6410256410256411
Retrying sample: 6778
Retry succeeded for sample: 6778
WER: 0.4594594594594595


In [ ]:
# ============================================================
# Check failed samples after retry
# ============================================================

failed_samples_after_retry = df_asr_results[
    (df_asr_results["asr_text_noisy"].isna()) |
    (df_asr_results["asr_text_noisy"].astype(str).str.strip() == "") |
    (df_asr_results["wer"].isna())
]

print("Number of failed samples after retry:", len(failed_samples_after_retry))

failed_samples_after_retry

Number of failed samples after retry: 0


,sample_index,original_text_for_asr,asr_text_noisy,noise_level,wer,clean_audio_path,noisy_audio_path


## 12. Create and Save the Final ASR Dataset

The corrected ASR results are joined with the original dataset and saved
to Google Drive as the final output file.

In [ ]:
# ============================================================
# Create final ASR dataset after retry and save to Drive
# ============================================================

# Sort results by sample index
df_asr_results = (
    df_asr_results
    .sort_values("sample_index")
    .reset_index(drop=True)
)

# Save the corrected progress file
df_asr_results.to_csv(progress_path, index=False)

print("Updated progress file saved:")
print(progress_path)

# Combine the original dataset with the corrected ASR outputs
df_asr_final = pd.concat(
    [
        df_asr.reset_index(drop=True),
        df_asr_results[
            [
                "original_text_for_asr",
                "asr_text_noisy",
                "noise_level",
                "wer",
                "clean_audio_path",
                "noisy_audio_path"
            ]
        ].reset_index(drop=True)
    ],
    axis=1
)

# Save final file to Google Drive
df_asr_final.to_csv(final_path, index=False)

print("\nSaved final file to Google Drive:")
print(final_path)
print("Final shape:", df_asr_final.shape)

Updated progress file saved:
/content/drive/MyDrive/MedAlert_ASR_outputs/asr_results_progress_8556_very_high_text_cleaned.csv

Saved final file to Google Drive:
/content/drive/MyDrive/MedAlert_ASR_outputs/ems_asr_noisy_dataset_8556_very_high_text_cleaned.csv
Final shape: (8556, 21)


## 13. Final Quality Checks

The final ASR dataset is checked for missing transcripts, empty transcripts, missing WER values, and average WER.

In [ ]:
# ============================================================
# Final quality checks
# ============================================================

print("Number of rows:", len(df_asr_final))
print("Missing ASR transcripts:", df_asr_final["asr_text_noisy"].isna().sum())
print(
    "Empty ASR transcripts:",
    (
        df_asr_final["asr_text_noisy"]
        .fillna("")
        .astype(str)
        .str.strip()
        == ""
    ).sum()
)
print("Missing WER values:", df_asr_final["wer"].isna().sum())

print("\nAverage WER:")
print(df_asr_final["wer"].dropna().mean())

print("\nNoise level distribution:")
print(df_asr_final["noise_level"].value_counts(dropna=False))

## 14. Download and Review Results

The final file is downloaded and selected examples are displayed for review.

In [ ]:
from google.colab import files

files.download(final_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_asr_final[["text_cleaned", "asr_text_noisy", "wer"]].head()

,text_cleaned,asr_text_noisy,wer
0,We are transporting an 80-year-old female with...,We are transferring an 80 year old female with...,0.473684
1,80-year-old female with recent sepsis and intu...,80-year-old female who's in substance and incu...,0.666667
2,We’re bringing in an 80-year-old female with a...,We're bringing in an 80-year-old female with a...,0.491803
3,80-year-old female with history of Sjogren’s a...,80-year-old female with history of children's ...,0.389610
4,We are transporting a 69-year-old male with a ...,We are transporting a 69-year-old male with a ...,0.357143


In [ ]:
# ============================================================
# Show one random ASR example without truncation
# ============================================================

import textwrap

# Choose one random example from the final ASR dataset
row = df_asr_final.sample(n=1, random_state=10).iloc[0]

line_width = 95

print("Source case ID:", row["source_case_id"])
print("Text variant:", row["text_variant"])
print("WER:", round(row["wer"], 3))
print("-" * 100)

print("\nOriginal cleaned text:")
print(
    textwrap.fill(
        str(row["original_text_for_asr"]),
        width=line_width
    )
)

print("\nASR noisy transcript:")
print(
    textwrap.fill(
        str(row["asr_text_noisy"]),
        width=line_width
    )
)

Source case ID: 32799170
Text variant: professional_complete
WER: 0.226
----------------------------------------------------------------------------------------------------

Original cleaned text:
We are transporting a 46-year-old female with a history of Crohn’s disease who is experiencing
sudden onset upper abdominal cramping radiating to her chest and back, consistent with her past
Crohn’s flares. She reports nausea and four episodes of bilious vomiting with small blood
specks but denies diarrhea or constipation. Her vitals are stable: heart rate 72, blood
pressure 132/65, respiratory rate 16, oxygen saturation 100%, and temperature 97.4. She has not
taken her Humira injection in over two weeks. Patient is currently stable.

ASR noisy transcript:
We are transporting a 46-year-old female with a history of proven disease who was experiencing
sudden onset of her abdominal cramping radiant or chest and back, consistent with her past
crows' flares. She reports nausea and four episodes of